# 02 — Data Quality Analysis

This notebook applies reusable validation checks to the normalized event dataset and exports an auditable cleaned dataset.

## Coverage
- Required schema and type validation
- Missing values and duplicate records
- Timestamp and stage checks
- Duplicate customer events
- Outlier and category checks
- Cleaning summary and cleaned-data export

In [ ]:
from pathlib import Path
import pandas as pd

ROOT = Path.cwd()
if not (ROOT / "data").exists() and (ROOT.parent / "data").exists():
    ROOT = ROOT.parent

DATA_PATH = ROOT / "data" / "processed" / "ecommerce_funnel_events_normalized.csv"
CLEAN_PATH = ROOT / "data" / "processed" / "ecommerce_funnel_events_clean.csv"
if not DATA_PATH.exists():
    raise FileNotFoundError(f"Run the modular pipeline first: {DATA_PATH}")

raw = pd.read_csv(DATA_PATH)
raw["event_timestamp"] = pd.to_datetime(raw["event_timestamp"], utc=True, errors="coerce")

required = {"event_id", "user_id", "session_id", "event_timestamp", "campaign", "channel", "device", "stage", "cart_value", "revenue"}
missing_columns = sorted(required - set(raw.columns))
if missing_columns:
    raise ValueError(f"Missing required columns: {missing_columns}")

clean = raw.drop_duplicates().copy()
clean = clean.loc[clean["event_timestamp"].notna()].copy()
clean = clean.loc[clean["stage"].isin(["landing", "product_view", "cart", "checkout", "purchase"])].copy()
clean = clean.drop_duplicates(subset=["event_id"])
clean = clean.sort_values(["user_id", "event_timestamp", "stage"])
CLEAN_PATH.parent.mkdir(parents=True, exist_ok=True)
clean.to_csv(CLEAN_PATH, index=False)

quality = pd.DataFrame([{
    "raw_rows": len(raw),
    "clean_rows": len(clean),
    "excluded_rows": len(raw) - len(clean),
    "duplicate_event_ids": int(raw["event_id"].duplicated().sum()),
    "invalid_timestamps": int(raw["event_timestamp"].isna().sum()),
}])
quality.to_csv(ROOT / "data" / "processed" / "data_quality_summary.csv", index=False)
display(quality)
print("Clean dataset:", CLEAN_PATH)


,raw_rows,clean_rows,excluded_rows,duplicate_event_ids,invalid_timestamps
0,2756101,2756101,0,0,0


Clean dataset: /home/essie/e-commerce-conversion-customer-funnel-analysis/data/processed/ecommerce_funnel_events_clean.csv
